In [ ]:
# install Google OR Tools
# %pip install ortools

In [17]:
# import modules
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

In [21]:
# stores the data for the problem
def create_data_model():
    data = {}
    data["distance_matrix"] = [
        [0, 2451, 713, 1018, 1631, 1374, 2408, 213, 2571, 875, 1420, 2145, 1972],
        [2451, 0, 1745, 1524, 831, 1240, 959, 2596, 403, 1589, 1374, 357, 579],
        [713, 1745, 0, 355, 920, 803, 1737, 851, 1858, 262, 940, 1453, 1260],
        [1018, 1524, 355, 0, 700, 862, 1395, 1123, 1584, 466, 1056, 1280, 987],
        [1631, 831, 920, 700, 0, 663, 1021, 1769, 949, 796, 879, 586, 371],
        [1374, 1240, 803, 862, 663, 0, 1681, 1551, 1765, 547, 225, 887, 999],
        [2408, 959, 1737, 1395, 1021, 1681, 0, 2493, 678, 1724, 1891, 1114, 701],
        [213, 2596, 851, 1123, 1769, 1551, 2493, 0, 2699, 1038, 1605, 2300, 2099],
        [2571, 403, 1858, 1584, 949, 1765, 678, 2699, 0, 1744, 1645, 653, 600],
        [875, 1589, 262, 466, 796, 547, 1724, 1038, 1744, 0, 679, 1272, 1162],
        [1420, 1374, 940, 1056, 879, 225, 1891, 1605, 1645, 679, 0, 1017, 1200],
        [2145, 357, 1453, 1280, 586, 887, 1114, 2300, 653, 1272, 1017, 0, 504],
        [1972, 579, 1260, 987, 371, 999, 701, 2099, 600, 1162, 1200, 504, 0],
    ]
    data["num_vehicles"] = 2
    data["depot"] = 0
    return data

In [22]:
# print the solution details        
def print_solution(manager, routing, solution):
    print(f"Objective: {solution.ObjectiveValue()} miles")
    index = routing.Start(0)
    plan_output = "Route for Vehicle 0: \n"
    route_distance = 0
    while not routing.IsEnd(index):
        plan_output += f" {manager.IndexToNode(index)} ->"
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        route_distance += routing.GetArcCostForVehicle(previous_index, index, 0)
    plan_output += f" {manager.IndexToNode(index)}\n"
    plan_output += f"Route Distance: {route_distance} miles"
    print(plan_output)

In [24]:
# main program
def main():
    # create the model
    data = create_data_model()
    # create the routing index manager
    manager = pywrapcp.RoutingIndexManager(
        len(data["distance_matrix"]), data["num_vehicles"], data["depot"],
    )
    # create the routing model
    routing = pywrapcp.RoutingModel(manager)
    # distance callback - takes any pair of locations and returns the distance between them
    def distance_callback(from_index, to_index):
        # convert from routing variable Index to distance matrix NodeIndex
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return data["distance_matrix"][from_node][to_node]
    # callback and registers it with the solver as transit_callback_index
    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    # set the cost of travel -> In this example, distance between them
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)
    # sets the first solution strategy to PATH_CHEAPEST_ARC, which creates 
    # an initial route for the solver by repeatedly adding edges with the least weight 
    # that don't lead to a previously visited node (other than the depot)
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = ( 
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC 
    )
    # run the solver
    solution = routing.SolveWithParameters(search_parameters)
    # print the solution
    if solution:
        print_solution(manager, routing, solution)

if __name__ == "__main__":
    main()

Objective: 7293 miles
Route for Vehicle 0: 
 0 -> 7 -> 2 -> 3 -> 4 -> 12 -> 6 -> 8 -> 1 -> 11 -> 10 -> 5 -> 9 -> 0
Route Distance: 7293 miles
